In [0]:
%python
print("Prepared by, Crystal Poon")

print("Project: Banking Transaction ETL")
print("Environment: Databricks Free Edition")
print("Language: Python")

In [0]:
%python
print("Spark Version:", spark.version)

Notes:
- Apache Spark = distributed data processing engine 
- Spark able to distribute data processing across compute resources instead of processing large datasets using ordinary Python operations

### Playground

In [0]:
%python
data = [
    (1001, "C001", 500.00, "TRANSFER"),
    (1002, "C002", 250.00, "PAYMENT"),
    (1003, "C003", 1000.00, "TRANSFER")
]

columns = ["transaction_id", "customer_id", "amount", "transaction_type"]

df = spark.createDataFrame(data, columns)

display(df)

In [0]:
%python
print("TEST 1: Count the transactions")

total_records = df.count()

print("Total transactions:", total_records)

assert total_records == 3, "Transaction count is incorrect"

print("TEST PASSED")

In [0]:
%python
print("TEST 2: Check for missing customer IDs")

from pyspark.sql.functions import col

missing_customer = df.filter(
    col("customer_id").isNull()
).count()

print("Missing customer IDs:", missing_customer)

assert missing_customer == 0, "Missing customer IDs found"

print("TEST PASSED")

In [0]:
%python
print("TEST 3: Check invalid transaction amounts")

invalid_amounts = df.filter(
    col("amount") <= 0
).count()

print("Invalid transaction amounts:", invalid_amounts)

assert invalid_amounts == 0, "Invalid amounts found"

print("TEST PASSED")

### Create the synthetic banking dataset

In [0]:
%python
print("Create the synthetic banking dataset")

from pyspark.sql import types as T
from pyspark.sql import functions as F

print("Libraries imported successfully")

In [0]:
%python
#StructField(Column name, Data type, Nullable)

transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), True),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("amount", T.StringType(), True),
    T.StructField("transaction_type", T.StringType(), True),
    T.StructField("transaction_date", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("ingestion_timestamp", T.StringType(), True)
])

print("Banking transaction_schema created")

In [0]:
%python
#simulate 20 synthetic records in list 
#(transaction_id, customer_id, amount, transaction_type, transaction_date, source_system, ingestion_timestamp)
transactions = [    
    ("TXN1001", "C001", "500.00", "TRANSFER", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1002", "C002", "250.00", "PAYMENT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1003", "C003", "1000.00", "TRANSFER", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1004", "C004", "75.00", "WITHDRAWAL", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1005", "C005", "120.00", "DEPOSIT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1006", "C006", "0.00", "PAYMENT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1007", None, "80.00", "TRANSFER", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1008", "C008", "-50.00", "PAYMENT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1009", "C009", "ABC", "TRANSFER", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1010", "C010", "45.00", "PAYMENT", "2026-13-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1011", "C011", "200.00", "DEPOSIT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1012", "C012", "150.00", "TRANSFER", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1013", "C013", "90.00", "WITHDRAWAL", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1014", "C014", "600.00", "PAYMENT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1015", "C015", "80.00", "TRANSFER", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1016", "C016", "110.00", "DEPOSIT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1017", "C017", "500.00", "PAYMENT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1018", "C018", "220.00", "TRANSFER", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),
    ("TXN1019", "C019", "320.00", "DEPOSIT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00"),

    # Test Duplicate transaction
    ("TXN1002", "C002", "250.00", "PAYMENT", "2026-11-01", "CORE_BANKING", "2026-11-02 01:00:00")
]

print("Total input records:", len(transactions))

In [0]:
%python
print("Convert transaction list into pyspark dataframe")
raw_df = spark.createDataFrame(transactions, schema=transaction_schema)
display(raw_df)

In [0]:
%python
print("Check schema")
raw_df.printSchema()

### Perform Data Profiling

In [0]:
%python
print("Test 1: Count transactions")
total_records = raw_df.count()

print("Total transactions:", total_records)

assert total_records == 20, "Transaction count is incorrect"

print("TEST PASSED")

In [0]:
%python
print("Test 2: Check for duplicate transactions")

from pyspark.sql import functions as F

duplicates_df = (raw_df
    .groupBy("transaction_id")
    .agg(F.count("*").alias("transaction_count"))
    .filter("transaction_count > 1")
)

dup_count = duplicates_df.count()

if dup_count > 0:
    display(duplicates_df)  # Only show results when duplicates exist

assert dup_count == 0, f"Found {dup_count} duplicate transaction ID(s)"
       

In [0]:
%python
print("Test 3: Check for invalid Customer id")

invalid_customers_df = raw_df.filter(
    F.col("customer_id").isNull() 
#    | (F.trim(F.col("customer_id")) == "")
#    | F.upper(F.trim(F.col("customer_id"))).isin("NULL", "N/A", "NONE", "UNKNOWN")
    | ~F.col("customer_id").rlike("^C[0-9]{3}$")  # IDs must follow a pattern like CXXX and already able to handle empty strings, "N/A", spaces, and wrong formats
)

missing_customer_count = invalid_customers_df.count()
print(f"Missing customer_id count: {missing_customer_count}")

if missing_customer_count > 0:
    #display(invalid_customers_df)  # or invalid_customers_df.show(truncate=False)
    print("Transaction ID(s) with invalid customer_id:")
    display(invalid_customers_df.select("transaction_id"))

assert missing_customer_count == 0, f"Test Failed: Found {missing_customer_count} invalid customer_id(s)"

print("TEST PASSED")

In PySpark, the tilde symbol (~) is the logical NOT operator used to negate a boolean column condition.

- `^C[0-9]+$` allows any number of digits after C (e.g., C1, C100, C999999)
- `^C[0-9]{3}$` allows only exact length like C001 (3 digits)


In [0]:
%python
from pyspark.sql import functions as F

print("Test 4: Check for invalid or non-numeric amount")

invalid_amount_df = raw_df.filter(
    F.col("amount").isNull()
    | F.expr("try_cast(amount as double)").isNull()  # Safely returns NULL on "ABC" under ANSI mode
)

invalid_amount_count = invalid_amount_df.count()
print(f"Invalid amount count: {invalid_amount_count}")

if invalid_amount_count > 0:
    display(invalid_amount_df.select("transaction_id", "amount"))

assert invalid_amount_count == 0, f"Test Failed: Found {invalid_amount_count} non-numeric amount(s)"

print("TEST PASSED")

### Load Data into Bronze delta table

In [0]:
Select current_catalog(), current_schema()

- Go to Catalog : Manually create new Catalog "banking_demo"
- Click on banking_demo, manually create 3 new schemas : bronze, silver, gold

In [0]:
USE CATALOG banking_demo;
USE SCHEMA bronze;

Select current_catalog(), current_schema()

In [0]:
%python
print("Set current location")
spark.catalog.setCurrentCatalog("banking_demo")
spark.catalog.setCurrentDatabase("bronze")  # "Database" is Spark's alias for Schema

print("Check current location")
print(f"Catalog: {spark.catalog.currentCatalog()}, Schema: {spark.catalog.currentDatabase()}")

In [0]:
%python
#double check table to populate into bronze table
display(raw_df)

In [0]:
%python
print("Create new Bronze table, transaction")
catalog_name = spark.catalog.currentCatalog()
schema_name = spark.catalog.currentDatabase()
bronze_table_name = "transaction"

raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{schema_name}.{bronze_table_name}")  

print("Table created")

In [0]:
select * from transaction;

In [0]:
%python
# Read the table
bronze_df = spark.read.table(f"{catalog_name}.{schema_name}.{bronze_table_name}")
display(bronze_df)
  

In [0]:
%python
# Validate bronze table
print("Test B1: Count of records in bronze table")

bronze_transaction_count = bronze_df.count()
print(f"Bronze transaction count: {bronze_transaction_count}")

assert bronze_transaction_count == 20, f"Test Failed: Found {bronze_transaction_count} records in bronze table"

print("TEST PASSED")

In [0]:
%python
# Validate bronze table
print("Test B2: Check for duplicate transactions")

from pyspark.sql import functions as F

duplicates_bronze_df = (bronze_df
    .groupBy("transaction_id")
    .agg(F.count("*").alias("transaction_count"))
    .filter("transaction_count > 1")
)

dup_count = duplicates_bronze_df.count()

if dup_count > 0:
    display(duplicates_bronze_df)  # Only show results when duplicates exist

assert dup_count == 0, f"Found {dup_count} duplicate transaction ID(s)"
       

In [0]:
%python
# Validate bronze table
print("Test B3: Check for invalid Customer id")

invalid_customers_df = bronze_df.filter(
    F.col("customer_id").isNull() 
#    | (F.trim(F.col("customer_id")) == "")
#    | F.upper(F.trim(F.col("customer_id"))).isin("NULL", "N/A", "NONE", "UNKNOWN")
    | ~F.col("customer_id").rlike("^C[0-9]{3}$")  # IDs must follow a pattern like CXXX and already able to handle empty strings, "N/A", spaces, and wrong formats
)

missing_customer_count = invalid_customers_df.count()
print(f"Missing customer_id count: {missing_customer_count}")

if missing_customer_count > 0:
    #display(invalid_customers_df)  # or invalid_customers_df.show(truncate=False)
    print("Transaction ID(s) with invalid customer_id:")
    display(invalid_customers_df.select("transaction_id"))

assert missing_customer_count == 0, f"Test Failed: Found {missing_customer_count} invalid customer_id(s)"

print("TEST PASSED")

In [0]:
%python
# Validation bronze table
from pyspark.sql import functions as F

print("Test 4: Check for invalid or non-numeric amount")

invalid_amount_df = bronze_df.filter(
    F.col("amount").isNull()
    | F.expr("try_cast(amount as double)").isNull()  # Safely returns NULL on "ABC" under ANSI mode
)

invalid_amount_count = invalid_amount_df.count()
print(f"Invalid amount count: {invalid_amount_count}")

if invalid_amount_count > 0:
    display(invalid_amount_df.select("transaction_id", "amount"))

assert invalid_amount_count == 0, f"Test Failed: Found {invalid_amount_count} non-numeric amount(s)"

print("TEST PASSED")

### Prepare Silver and Rejected schema

In [0]:
USE CATALOG banking_demo;
SHOW SCHEMAS IN banking_demo;

In [0]:
CREATE SCHEMA IF NOT EXISTS banking_demo.rejected;
SHOW SCHEMAS;

In [0]:
%python
bronze_df = spark.read.table("banking_demo.bronze.transaction")
print(f" {bronze_df.count()} records in bronze table")

display(bronze_df)

In [0]:
%python
print("Prepare for silver table")

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries imported successfully")

- Bronze table store data as string
- Silver table store based on the nature of the data

In [0]:
%python
bronze_df.printSchema()

bronze.transaction
 - transaction_id: string (nullable = true)
 - customer_id: string (nullable = true)
 - amount: string (nullable = true)
 - transaction_type: string (nullable = true)
 - transaction_date: string (nullable = true)
 - source_system: string (nullable = true)
 - ingestion_timestamp: string (nullable = true)

silver.transaction
 - transaction_id: string (nullable = false)
 - customer_id: string (nullable = false)
 - amount: decimal(18,2) (nullable = true)
 - transaction_type: string (nullable = false)
 - transaction_date: date (nullable = false)
 - source_system: string (nullable = true)
 - ingestion_timestamp: string (nullable = true)

In [0]:
%python
print("Converting invalid or non-numeric amount")

staged_df = (bronze_df
    .withColumn("amount_decimal",
                F.expr("try_cast(amount as decimal(18,2))"))
    .withColumn("transaction_date_parsed",
                F.expr("try_cast(transaction_date as date)"))
    .withColumn("transaction_type_cleaned",
                F.upper(F.trim(F.col("transaction_type"))))
)
display(staged_df)
staged_df.printSchema()


In [0]:
%python
print("Checking converted data..")
staged_df.select("transaction_id",
                 "amount",
                 "amount_decimal",
                 "transaction_date",
                 "transaction_date_parsed",
                 "transaction_type_cleaned"
                 ).show(truncate=False)
       

In [0]:
%python
# identify duplicate: grouping duplicate records together and sorting them chronologically from oldest to newest.
duplicate_window = Window.partitionBy("transaction_id").orderBy(F.col("ingestion_timestamp").asc())

staged_df = staged_df.withColumn("duplicate_rank", 
                                 F.row_number().over(duplicate_window))

staged_df.show(truncate=False)

In [0]:
%python
#only show rank > 1 for duplicate records
duplicate_records = (staged_df.filter(F.col("duplicate_rank") > 1))
display(duplicate_records)

In [0]:
%python
allowed_transaction_types = ["TRANSFER", "PAYMENT", "WITHDRAWAL", "DEPOSIT"]

validated_df = staged_df.withColumn("rejection_reason",
            F.concat_ws("|",
                        F.when(F.col("transaction_id").isNull() | (F.trim(F.col("transaction_id"))==""),
                               F.lit("MISSING_TRANSACTION_ID")),
                        F.when(F.col("customer_id").isNull() | (F.trim(F.col("customer_id"))==""),
                               F.lit("MISSING_CUSTOMER_ID")),
                        F.when(F.col("amount_decimal").isNull(),
                               F.lit("INVALID_AMOUNT")),
                        F.when(F.col("amount_decimal") <= 0,
                               F.lit("NON_POSITIVE_AMOUNT")),
                        F.when(~F.col("transaction_type_cleaned").isin(allowed_transaction_types),
                               F.lit("INVALID_TRANSACTION_TYPE")),
                        F.when(F.col("transaction_date_parsed").isNull(),
                               F.lit("INVALID_TRANSACTION_DATE")),
                        F.when(F.col("duplicate_rank")>1,
                               F.lit("DUPLICATE_TRANSACTION_ID"))
                        ))

display(validated_df)

Separate the good and bad records

In [0]:
%python
# filter out the bad records
silver_df = (validated_df.filter(F.col("rejection_reason")=="")
             .select(
                 "transaction_id",
                 "customer_id",
                 F.col("amount_decimal").alias("amount"),
                 F.col("transaction_type_cleaned").alias("transaction_type"),
                 F.col("transaction_date_parsed").alias("transaction_date"),
                 "source_system",
                 "ingestion_timestamp"
             )
            )

print("Number of records in silver table: ", silver_df.count())

silver_df.show(truncate=False)

In [0]:
%python
# filter out the bad records
rejected_df = (validated_df.filter(F.col("rejection_reason")!="")
             .select(
                 "transaction_id",
                 "customer_id",
                 "amount",
                 "transaction_type",
                 "transaction_date",
                 "source_system",
                 "ingestion_timestamp",
                 "rejection_reason"
             )
            )

print("Number of records in rejected table: ", rejected_df.count())

rejected_df.show(truncate=False)

In [0]:
%python
print("Rejected reason report")

reject_report_df = (validated_df.filter(F.col("rejection_reason")!="")
                    .select("rejection_reason")
                    .groupBy("rejection_reason")
                    .count()
                    .orderBy(F.col("rejection_reason").asc())
                   )
display(reject_report_df)

### Save into silver and rejected table

In [0]:
%python
silver_df.write.format("delta").mode("overwrite").saveAsTable("banking_demo.silver.transaction")
print("Number of records in silver table: ", spark.table("banking_demo.silver.transaction").count())

rejected_df.write.format("delta").mode("overwrite").saveAsTable("banking_demo.rejected.transaction")
print("Number of records in rejected table: ", spark.table("banking_demo.rejected.transaction").count())


In [0]:
SELECT * FROM banking_demo.silver.transaction;

In [0]:
SELECT * FROM banking_demo.rejected.transaction;

In [0]:
SELECT rejection_reason, transaction_id 
FROM banking_demo.rejected.transaction 
ORDER BY rejection_reason, transaction_id;

### Perform Data Reconciliation

bronze records count = silver records count + rejected records count

In [0]:
%python
print("Validating data reconciliation check")
bronze_rec_count = spark.table("banking_demo.bronze.transaction").count()
print("Number of records in bronze table: ", bronze_rec_count)

silver_rec_count = spark.table("banking_demo.silver.transaction").count()
print("Number of records in silver table: ", silver_rec_count)

rejected_rec_count = spark.table("banking_demo.rejected.transaction").count()
print("Number of records in rejected table: ", rejected_rec_count)

assert bronze_rec_count == silver_rec_count + rejected_rec_count, "Data reconciliation check failed"

print("Data reconciliation check PASSED")


### Additional tests

In [0]:
%python
from pyspark.sql import functions as F

In [0]:
%python
print("Test S1: Validate silver data types")

spark.table("banking_demo.silver.transaction").printSchema()

assert spark.table("banking_demo.silver.transaction").schema["amount"].dataType.typeName() == "decimal", "Data type mismatch for amount"

assert spark.table("banking_demo.silver.transaction").schema["transaction_date"].dataType.typeName() == "date", "Data type mismatch for transaction_date"

print("Test S1: PASSED")

In [0]:
%python
print("Test S2: Validate no duplicates of transactions in silver table")

duplicate_count = (spark.table("banking_demo.silver.transaction")
                   .groupBy("transaction_id").count()
                   .filter(F.col("count")>1).count())

assert duplicate_count == 0, "Duplicate records found in silver table"

print("Test S2: PASSED")

In [0]:
%python
print("Test S3: Validate amount is in positive value in silver table")

non_positive_amt_count = (spark.table("banking_demo.silver.transaction")
                          .filter(F.col("amount")<=0).count())

assert non_positive_amt_count == 0, "Negative or zero amount found in silver table"

print("Test S3: PASSED")

In [0]:
%python
print("Test S4: Validate date is valid in silver table")

invalid_date_count = (spark.table("banking_demo.silver.transaction")
                      .filter(
                          F.col("transaction_date").isNull() | 
                          F.try_to_date(F.col("transaction_date"), "yyyy-MM-dd").isNull())
                      .count()
)

assert invalid_date_count == 0, "Invalid date found in silver table"

print("Test S4: PASSED")